In [4]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd
import time
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path Dataset
input_file = "/content/drive/MyDrive/data_clean/hasil_stemming.csv"
output_file = "/content/drive/MyDrive/data_clean/hasil_translation.csv"

# Gunakan Model yang Lebih Ringan
MODEL_NAME = "Helsinki-NLP/opus-mt-id-en"

print(f"Loading model {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Fix error dengan menghapus device_map="auto"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,  # Optimasi GPU
).to(device)

# Fungsi Terjemahan
def translate_to_english(text):
    if isinstance(text, list):
        text = " ".join(text)

    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=200,
            pad_token_id=tokenizer.eos_token_id
        )

    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translation

# Fungsi Batch Processing
def translate_dataset_in_batches(df, column_name, batch_size=2):
    results = []
    total_batches = (len(df) + batch_size - 1) // batch_size

    for i in tqdm(range(0, len(df), batch_size), total=total_batches, desc="Translating"):
        batch = df[column_name].iloc[i:i+batch_size]
        batch_results = []

        for text in batch:
            translation = translate_to_english(text)
            batch_results.append(translation)
            time.sleep(0.05)  # Kurangi delay

        results.extend(batch_results)

        # Clear CUDA cache jika pakai GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

# Fungsi untuk Terjemahkan Dataset
def translate_dataset(input_file, output_file, column_to_translate='Stemming'):
    df = pd.read_csv(input_file)

    print(f"Loaded dataset with {len(df)} rows")

    # Proses Terjemahan
    df['EnglishTranslation'] = translate_dataset_in_batches(df, column_to_translate)

    # Simpan Hasil
    df.to_csv(output_file, index=False)
    print(f"Translations saved to {output_file}")

# Jalankan Proses
translate_dataset(input_file, output_file)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model Helsinki-NLP/opus-mt-id-en...


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Loaded dataset with 5000 rows


Translating: 100%|██████████| 2500/2500 [10:35:57<00:00, 15.26s/it]


Translations saved to /content/drive/MyDrive/data_clean/hasil_translation.csv


In [ ]:
output_file = "/content/drive/MyDrive/data_clean/translation_english.csv"
df = pd.read_csv(output_file)
df.head()  # Menampilkan 5 baris pertama


,userName,score,at,content,No Mentions & Hashtags,No URLs,No Numbers,No Special Characters,No Emojis,No Repeated Characters,Case Folding,Tokenizing,Formalisasi,WithoutStopwords,Stemming,EnglishTranslation
0,Pengguna Google,1,2025-03-15 13:05:26,Shopee yang sekarang kualitasnya bagus tetapi ...,Shopee yang sekarang kualitasnya bagus tetapi ...,Shopee yang sekarang kualitasnya bagus tetapi ...,Shopee yang sekarang kualitasnya bagus tetapi ...,Shopee yang sekarang kualitasnya bagus tetapi ...,Shopee yang sekarang kualitasnya bagus tetapi ...,Shopee yang sekarang kualitasnya bagus tetapi ...,shopee yang sekarang kualitasnya bagus tetapi ...,"['shopee', 'yang', 'sekarang', 'kualitasnya', ...","['shopee', 'yang', 'sekarang', 'kualitasnya', ...","['shopee', 'kualitasnya', 'bagus', 'parahnya',...","['shopee', 'kualitas', 'bagus', 'parah', 'kiri...","['shopee', 'quality', 'good', 'bad', 'send', '..."
1,Pengguna Google,4,2025-03-16 04:29:12,Apk belanja yang mudah dan beragam pilihan tok...,Apk belanja yang mudah dan beragam pilihan tok...,Apk belanja yang mudah dan beragam pilihan tok...,Apk belanja yang mudah dan beragam pilihan tok...,Apk belanja yang mudah dan beragam pilihan tok...,Apk belanja yang mudah dan beragam pilihan tok...,Apk belanja yang mudah dan beragam pilihan tok...,apk belanja yang mudah dan beragam pilihan tok...,"['apk', 'belanja', 'yang', 'mudah', 'dan', 'be...","['aplikasi', 'belanja', 'yang', 'mudah', 'dan'...","['apk', 'belanja', 'mudah', 'beragam', 'piliha...","['apk', 'belanja', 'mudah', 'agam', 'pilih', '...","['apk', 'shop', 'empty', 'agam', 'choke', 'tru..."
2,Pengguna Google,1,2025-03-15 14:16:55,"Tidak bisa melakukan pembayaran listrik, dan p...","Tidak bisa melakukan pembayaran listrik, dan p...","Tidak bisa melakukan pembayaran listrik, dan p...","Tidak bisa melakukan pembayaran listrik, dan p...",Tidak bisa melakukan pembayaran listrik dan p...,Tidak bisa melakukan pembayaran listrik dan p...,Tidak bisa melakukan pembayaran listrik dan p...,tidak bisa melakukan pembayaran listrik dan p...,"['tidak', 'bisa', 'melakukan', 'pembayaran', '...","['tidak', 'bisa', 'melakukan', 'pembayaran', '...","['pembayaran', 'listrik', 'pembayaran', 'tulis...","['bayar', 'listrik', 'bayar', 'tulis', 'downlo...","['pay', 'electric', 'pay', 'write', 'singing',..."
3,Pengguna Google,1,2025-03-15 05:13:35,"Sudah dua bulan, setiap buka langsung diarahin...","Sudah dua bulan, setiap buka langsung diarahin...","Sudah dua bulan, setiap buka langsung diarahin...","Sudah dua bulan, setiap buka langsung diarahin...",Sudah dua bulan setiap buka langsung diarahin...,Sudah dua bulan setiap buka langsung diarahin...,Sudah dua bulan setiap buka langsung diarahin...,sudah dua bulan setiap buka langsung diarahin...,"['sudah', 'dua', 'bulan', 'setiap', 'buka', 'l...","['sudah', 'dua', 'bulan', 'setiap', 'buka', 'l...","['buka', 'langsung', 'diarahin', 'live', 'vide...","['buka', 'langsung', 'diarahin', 'live', 'vide...","['open', 'direct', 'directin', 'live', 'video'..."
4,Pengguna Google,1,2025-03-15 08:03:40,"Aplikasi paling ga jelas, hp ku restart ulang ...","Aplikasi paling ga jelas, hp ku restart ulang ...","Aplikasi paling ga jelas, hp ku restart ulang ...","Aplikasi paling ga jelas, hp ku restart ulang ...",Aplikasi paling ga jelas hp ku restart ulang ...,Aplikasi paling ga jelas hp ku restart ulang ...,Aplikasi paling ga jelas hp ku restart ulang ...,aplikasi paling ga jelas hp ku restart ulang ...,"['aplikasi', 'paling', 'ga', 'jelas', 'hp', 'k...","['aplikasi', 'paling', 'ga', 'jelas', 'seluler...","['aplikasi', 'ga', 'hp', 'ku', 'restart', 'ula...","['aplikasi', 'ga', 'hp', 'ku', 'restart', 'ula...","['application', 'ga', 'hp', 'ku', 'restart', '..."
